# Initialization

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import unicodedata
from pyspark.sql.functions import col, sum, when, upper, trim

In [0]:
def to_snake_case(column_name):
    column_name = unicodedata.normalize("NFKD", column_name)
    column_name = column_name.encode("ascii", "ignore").decode("utf-8")
    column_name = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", column_name)
    column_name = re.sub(r"[^a-zA-Z0-9]+", "_", column_name)
    column_name = re.sub(r"_+", "_", column_name)
    column_name = column_name.strip("_")

    return column_name.lower()

In [0]:
def validate_codes(df, col_name, dim_table, dim_code="codigo"):
    dim = spark.table(dim_table)

    invalid = df.join(
        dim,
        df[col_name] == dim[dim_code],
        "left_anti"
    )

    print(f"{col_name}: {invalid.count()} invalid values")

    return invalid

# Visualitation

In [0]:
df = spark.table("covid19.bronze.ine_deaths")
print(df) 

In [0]:
df.head()

# Cleaning

## Columns normalitation

In [0]:
df = df.toDF(*[to_snake_case(col) for col in df.columns])

## Delete exactly duplicate rows

In [0]:
# delete exact duplicate rows
total_rows = df.count()
unique_rows = df.dropDuplicates().count()

print(f"Total filas: {total_rows}")
print(f"Filas únicas: {unique_rows}")
print(f"Duplicados: {total_rows - unique_rows}")

## delete nulls 

In [0]:


null_counts = df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
])

display(null_counts)

In [0]:
total_rows = df.count()

for column in df.columns:
    nulls = df.filter(col(column).isNull()).count()
    percentage = (nulls / total_rows) * 100

    print(
        f"{column}: {nulls} nulls ({percentage:.2f}%)"
    )

In [0]:
df = df.fillna({
    "Escodif": 99,
    "Ciuodif": 99
})

## correct data types

In [0]:
df.printSchema()

## standarize 

In [0]:
display(
    df.select("Caudef")
      .distinct()
      .orderBy("Caudef")
)

In [0]:

df = df.withColumn(
    "Caudef",
    upper(trim("Caudef"))
)

## semantic validation

In [0]:
display(df.select("Sexo").distinct())
display(df.select("Mesreg").distinct())
display(df.select("Edadif").describe())

In [0]:
df = df.withColumn(
    "Edadif",
    when(col("Edadif") == 999, None)
    .otherwise(col("Edadif"))
)

In [0]:
df.filter(df.Edadif < 0).count()

In [0]:
df = df.filter((df.Edadif >= 0) | df.Edadif.isNull())

## delete outlayers

In [0]:
q1, q3 = df.approxQuantile("Edadif", [0.25, 0.75], 0.01)

iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

print(lower, upper)

In [0]:
df_clean = df.filter(
    (df.Edadif.isNull()) |
    ((df.Edadif >= lower) & (df.Edadif <= upper))
)